# Model registry and serving

Once you've found a model that actually works, you need a way to manage its lifecycle. In this notebook, we'll cover:

1. Picking the best run from our previous experiments.
2. Registering it in the **MLflow Model Registry** so we can track versions.
3. Loading the model for inference using framework-agnostic `mlflow.pyfunc`.
4. Serving the model as a REST API or inside a Docker container.

## 1. Setup and imports

In [1]:
import sys, os

REPO_ROOT = os.path.abspath(os.pardir)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import mlflow
from mlflow import MlflowClient

from src.data_preprocessing import load_data, preprocess, split_data

print("MLflow version:", mlflow.__version__)

mlflow.set_tracking_uri(f"sqlite:///{REPO_ROOT}/mlflow.db")

MLflow version: 3.15.2


## 2. Finding the best run

We'll use `mlflow.search_runs()` to find the run with the highest F1 score in the `telco-churn` experiment.

In [3]:
experiment = mlflow.get_experiment_by_name("telco-churn-experiments")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"],
)

best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_run_name = best_run.get("tags.mlflow.runName", "unknown")

print(f"Best run: {best_run_name}")
print(f"Run ID:   {best_run_id}")
print(f"F1 Score: {best_run['metrics.f1_score']:.4f}")
print(f"ROC AUC:  {best_run['metrics.roc_auc']:.4f}")

Best run: xgb-n100-d4-lr0.1
Run ID:   ef143920b1174580bace14e5c1e7323b
F1 Score: 0.7260
ROC AUC:  0.7219


## 3. Registering the model

The **Model Registry** is a central place to store and version your models. It's better than just keeping track of run IDs because you can:

- Version your models (v1, v2, etc.).
- Add descriptions or tags like `approved` or `staged`.
- Use aliases like `champion` to point to the current production model.

In [4]:
MODEL_NAME = "TelcoChurnModel"

# Register the model from the best run
model_uri = f"runs:/{best_run_id}/xgb-model"
result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"Registered model: {result.name}")
print(f"Version:          {result.version}")
print(f"Source:           {result.source}")

Registered model 'TelcoChurnModel' already exists. Creating a new version of this model...
2026/09/02 12:46:40 WARNING mlflow.tracking._model_registry.fluent: Run with id ef143920b1174580bace14e5c1e7323b has no artifacts at artifact path 'xgb-model', registering model based on models:/m-3e3e158bbadf4385b6b08dbf9d74214d instead
Created version '2' of model 'TelcoChurnModel'.


Registered model: TelcoChurnModel
Version:          2
Source:           models:/m-3e3e158bbadf4385b6b08dbf9d74214d


## 4. Managing model versions with `MlflowClient`

The `MlflowClient` gives you more control over the registry. We can use it to update descriptions, set tags, and manage aliases.

In [5]:
client = MlflowClient()

# Add a description to the model version
client.update_model_version(
    name=MODEL_NAME,
    version=result.version,
    description=(
        f"Best model from experiment tracking (run: {best_run_name}). "
        f"F1={best_run['metrics.f1_score']:.4f}, AUC={best_run['metrics.roc_auc']:.4f}."
    ),
)

# Set tags on the model version
client.set_model_version_tag(
    name=MODEL_NAME,
    version=result.version,
    key="validation_status",
    value="approved",
)

print(f"Updated description and tags for {MODEL_NAME} v{result.version}")

Updated description and tags for TelcoChurnModel v2


### 4.1 Assigning aliases

Instead of keeping track of version numbers, you can use aliases like `champion`. This makes it much easier to swap out models in production without changing your code.

In [6]:
# Set the 'champion' alias to point to our best version
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=result.version,
)

print(f"Alias 'champion' now points to {MODEL_NAME} v{result.version}")

Alias 'champion' now points to TelcoChurnModel v2


In [7]:
# List all versions of the registered model
print(f"\nAll versions of '{MODEL_NAME}':")
for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
    aliases = mv.aliases if hasattr(mv, "aliases") else []
    print(
        f"  Version {mv.version} | "
        f"Run ID: {mv.run_id[:8]}... | "
        f"Aliases: {aliases}"
    )


All versions of 'TelcoChurnModel':
  Version 2 | Run ID: ef143920... | Aliases: []
  Version 1 | Run ID: ef143920... | Aliases: []


## 5. Loading a model from the registry

You can load a model by its version number or its alias. Using an alias is generally better for production code.

| URI format | Example |
|---|---|
| By version | `models:/TelcoChurnModel/1` |
| By alias | `models:/TelcoChurnModel@champion` |

In [8]:
# Load the 'champion' model
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print(f"Loaded model type: {type(champion_model)}")
print("This is a framework-agnostic PyFunc wrapper.")

Loaded model type: <class 'mlflow.pyfunc.PyFuncModel'>
This is a framework-agnostic PyFunc wrapper.


### 5.2 Loading by version

In [9]:
# Load a specific version
model_v1 = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/1")
print(f"Loaded model v1: {type(model_v1)}")

Loaded model v1: <class 'mlflow.pyfunc.PyFuncModel'>


## 6. Making predictions

Now that we've loaded the model, let's use it on our test set.

In [10]:
# Prepare test data
df = load_data()
X, y, scaler, feature_names = preprocess(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Scoring {len(X_test)} samples...")

# Predict using the champion model (framework-agnostic via pyfunc)
predictions = champion_model.predict(X_test)

# Show a sample of predictions alongside actuals
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": predictions,
})

print(f"\nSample predictions:")
results.head(10)

Scoring 20000 samples...

Sample predictions:


,actual,predicted
0,1,1
1,0,0
2,0,0
3,1,0
4,0,0
5,1,1
6,0,0
7,1,0
8,1,0
9,0,0


In [11]:
from sklearn.metrics import accuracy_score, f1_score

print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(f"F1 Score: {f1_score(y_test, predictions):.4f}")

Accuracy: 0.6807
F1 Score: 0.7240


## 7. Serving the model as an API

You can also serve your model as a REST API. Run this in your terminal:

```bash
mlflow models serve -m "models:/TelcoChurnModel@champion" --port 5001 --no-conda
```

Then you can send a prediction request with `curl`. Just make sure the feature names in your JSON match what the model expects.

In [12]:
import json

# Build a single-row payload from X_test (columns and data must match model input)
sample_row = X_test.iloc[:1]
payload = {
    "dataframe_split": {
        "columns": list(sample_row.columns),
        "data": sample_row.values.tolist(),
    }
}
payload_json = json.dumps(payload)

print("Run this curl after starting the model server (port 5001):\n")
print(f"curl -X POST http://localhost:5001/invocations \\")
print(f"  -H 'Content-Type: application/json' \\")
print(f"  -d '{payload_json}'")

Run this curl after starting the model server (port 5001):

curl -X POST http://localhost:5001/invocations \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {"columns": ["gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService", "PaperlessBilling", "MonthlyCharges", "TotalCharges", "MultipleLines_No phone service", "MultipleLines_Yes", "InternetService_Fiber optic", "InternetService_No", "OnlineSecurity_No internet service", "OnlineSecurity_Yes", "OnlineBackup_No internet service", "OnlineBackup_Yes", "DeviceProtection_No internet service", "DeviceProtection_Yes", "TechSupport_No internet service", "TechSupport_Yes", "StreamingTV_No internet service", "StreamingTV_Yes", "StreamingMovies_No internet service", "StreamingMovies_Yes", "Contract_One year", "Contract_Two year", "PaymentMethod_Credit card (automatic)", "PaymentMethod_Electronic check", "PaymentMethod_Mailed check"], "data": [[0, 0, 0, 1, -0.04637780884491682, 1, 0, -0.05478560188861699, -0

## 8. Building a Docker image for deployment

If you want to deploy the model to a cloud environment, you can package it into a Docker image. This image includes the model and everything needed to run a REST API server.

### Using the Python API

```python
import mlflow

model_uri = "models:/TelcoChurnModel@champion"
image_name = "telco-churn-model:latest"

# Build the image (port 8080 by default)
mlflow.models.build_docker(model_uri=model_uri, name=image_name)
```

### Using the CLI

```bash
mlflow models build-docker -m "models:/TelcoChurnModel@champion" -n telco-churn-model:latest
```

Once it's built, you can run the container locally with:

```bash
docker run -p 8080:8080 telco-churn-model:latest
```

In [14]:
import mlflow

model_uri = "models:/TelcoChurnModel@champion"
image_name = "telco-churn-model:latest"

mlflow.models.build_docker(model_uri=model_uri, name=image_name)

2026/09/02 13:09:19 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/09/02 13:09:19 INFO mlflow.pyfunc.backend: Building docker image with name telco-churn-model:latest
#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 2.19kB done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/ubuntu:22.04
#2 DONE 2.1s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 DONE 0.0s

#5 [ 1/12] FROM docker.io/library/ubuntu:22.04@sha256:2edbbc5dc405e9612ba3584ce95480277e3eb374407b5505fe26f17df77c7dbc
#5 resolve docker.io/library/ubuntu:22.04@sha256:2edbbc5dc405e9612ba3584ce95480277e3eb374407b5505fe26f17df77c7dbc 0.0s done
#5 ...

#4 [internal] load build context
#4 transferring context: 169.63kB 0.0s done
#4 DONE 0.1s

#5 [ 1/12] FROM docker.io/library/ubuntu:22.04@sha256:2edbbc5dc405e9612ba358

In [13]:
# Uncomment to build the Docker image (requires Docker daemon, may take a few minutes):
# mlflow.models.build_docker(
#     model_uri=f"models:/{MODEL_NAME}@champion",
#     name="telco-churn-model:latest",
# )